# 04 – Clinical Variable Extraction (17 variables)

Extracts the benchmark 17 physiological variables from CHARTEVENTS and LABEVENTS for the
cohort ICU stays, using the itemid-to-variable mapping.

**Run after notebook 02** — uses final_cohort_angus.csv, raw CHARTEVENTS/LABEVENTS,
D_ITEMS, and itemid_to_variable_map.csv.

**Produces:** filtered_chartevents_17vars.csv and filtered_labevents_17vars.csv
(used by notebook 05).

MIMIC-III data not included (PhysioNet DUA); see README. Patient-row outputs cleared.
CHARTEVENTS and LABEVENTS are processed in chunks to reduce memory usage.

In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import pandas as pd

In [ ]:
mapping_path = data_path("itemid_to_variable_map.csv")

mapping = pd.read_csv(mapping_path)

print(mapping.shape)
print(mapping.columns.tolist())
display(mapping.head(20))

In [ ]:
for col in mapping.columns:
    print("\n", col)
    print(mapping[col].dropna().astype(str).unique()[:30])

In [ ]:
cohort_path = data_path("final_cohort_angus.csv")

cohort = pd.read_csv(cohort_path)

print(cohort.shape)
print(cohort[["HADM_ID", "ICUSTAY_ID", "Sepsis_Angus"]].head())

In [ ]:
ditems = pd.read_csv(
    data_path("D_ITEMS.csv.gz")
)

print(ditems.shape)
ditems.head()

In [ ]:
ditems.columns.tolist()

In [ ]:
vital_keywords = [
    "Heart",
    "Resp",
    "Temperature",
    "Temp",
    "SpO2",
    "O2",
    "Blood",
    "Pressure",
    "Arterial",
    "Mean",
    "MAP",
    "Systolic",
    "Diastolic"
]

vitals = ditems[
    ditems["LABEL"].str.contains(
        "|".join(vital_keywords),
        case=False,
        na=False
    )
]

print(vitals.shape)
vitals[
    ["ITEMID", "LABEL", "CATEGORY", "UNITNAME"]
].sort_values("LABEL")

In [ ]:
char_items = ditems[
    ditems["LINKSTO"] == "chartevents"
]

print(char_items.shape)

In [ ]:
vitals = char_items[
    char_items["LABEL"].str.contains(
        "|".join(vital_keywords),
        case=False,
        na=False
    )
]

vitals[
    ["ITEMID", "LABEL", "CATEGORY", "UNITNAME"]
].sort_values("LABEL")

In [ ]:
mapping_path = data_path("itemid_to_variable_map.csv")

mapping = pd.read_csv(mapping_path)

benchmark_17 = [
    "Capillary refill rate",
    "Diastolic blood pressure",
    "Fraction inspired oxygen",
    "Glascow coma scale eye opening",
    "Glascow coma scale motor response",
    "Glascow coma scale total",
    "Glascow coma scale verbal response",
    "Glucose",
    "Heart Rate",
    "Height",
    "Mean blood pressure",
    "Oxygen saturation",
    "Respiratory rate",
    "Systolic blood pressure",
    "Temperature",
    "Weight",
    "pH"
]

selected_map = mapping[
    mapping["LEVEL2"].isin(benchmark_17)
].copy()

# drop mappings the official map marks as 'ignore'
selected_map = selected_map[
    selected_map["STATUS"] != "ignore"
].copy()

selected_map["ITEMID"] = pd.to_numeric(
    selected_map["ITEMID"],
    errors="coerce"
)

selected_map = selected_map.dropna(
    subset=["ITEMID", "LINKSTO", "LEVEL2"]
)

selected_map["ITEMID"] = selected_map["ITEMID"].astype(int)

print("Selected mapping rows:", len(selected_map))

print("\nVariables found:")
print(
    selected_map["LEVEL2"]
    .value_counts()
    .sort_index()
)

print("\nSource tables:")
print(
    pd.crosstab(
        selected_map["LEVEL2"],
        selected_map["LINKSTO"]
    )
)

In [ ]:
char_map = selected_map[
    selected_map["LINKSTO"] == "chartevents"
].copy()

lab_map = selected_map[
    selected_map["LINKSTO"] == "labevents"
].copy()

char_itemids = set(
    char_map["ITEMID"].astype(int)
)

lab_itemids = set(
    lab_map["ITEMID"].astype(int)
)

print("CHARTEVENTS ITEMIDs:", len(char_itemids))
print("LABEVENTS ITEMIDs:", len(lab_itemids))

print("\nCHARTEVENTS variables:")
print(sorted(char_map["LEVEL2"].unique()))

print("\nLABEVENTS variables:")
print(sorted(lab_map["LEVEL2"].unique()))

In [ ]:
cohort_icu = set(
    cohort["ICUSTAY_ID"]
    .dropna()
    .astype(int)
)

print("Number of ICU stays:", len(cohort_icu))

In [ ]:
char_itemid_to_variable = dict(
    zip(
        char_map["ITEMID"],
        char_map["LEVEL2"]
    )
)

lab_itemid_to_variable = dict(
    zip(
        lab_map["ITEMID"],
        lab_map["LEVEL2"]
    )
)

print(list(char_itemid_to_variable.items())[:10])
print(list(lab_itemid_to_variable.items())[:10])

In [ ]:
CHAR_PATH = data_path("CHARTEVENTS.csv.gz")

CHAR_OUT = data_path("filtered_chartevents_17vars.csv")

if os.path.exists(CHAR_OUT):

    os.remove(CHAR_OUT)

first_write = True

total_kept = 0

total_processed = 0

reader = pd.read_csv(
    CHAR_PATH,
    usecols=[
        "ICUSTAY_ID",
        "ITEMID",
        "CHARTTIME",
        "VALUENUM",
        "VALUE",
        "VALUEUOM",
        "ERROR"
    ],
    chunksize=50_000,
    low_memory=False
)

for i, chunk in enumerate(reader, start=1):

    total_processed += len(chunk)

    chunk = chunk.dropna(
        subset=["ICUSTAY_ID", "ITEMID", "CHARTTIME"]
    ).copy()

    chunk["ICUSTAY_ID"] = pd.to_numeric(
        chunk["ICUSTAY_ID"],
        errors="coerce"
    )

    chunk["ITEMID"] = pd.to_numeric(
        chunk["ITEMID"],
        errors="coerce"
    )

    chunk = chunk.dropna(
        subset=["ICUSTAY_ID", "ITEMID"]
    ).copy()

    chunk["ICUSTAY_ID"] = chunk["ICUSTAY_ID"].astype("int64")
    chunk["ITEMID"] = chunk["ITEMID"].astype("int64")

    chunk = chunk[
        chunk["ICUSTAY_ID"].isin(cohort_icu)
        & chunk["ITEMID"].isin(char_itemids)
    ].copy()

    if "ERROR" in chunk.columns:
        chunk = chunk[
            chunk["ERROR"].isna()
            | (chunk["ERROR"] == 0)
        ].copy()

    if not chunk.empty:
        chunk["VARIABLE"] = chunk["ITEMID"].map(
            char_itemid_to_variable
        )

        chunk.to_csv(
            CHAR_OUT,
            mode="a",
            header=first_write,
            index=False
        )

        first_write = False
        total_kept += len(chunk)

    print(
        f"Chunk {i:03d} | "
        f"processed {total_processed:,} | "
        f"kept {total_kept:,}"
    )


print("\nFinished extraction.")
print("Total processed:", total_processed)
print("Total kept:", total_kept)
print("Output size GB:", os.path.getsize(CHAR_OUT) / 1024**3)

In [ ]:
print(os.path.exists(CHAR_OUT))

check = pd.read_csv(
    CHAR_OUT,
    usecols=["VARIABLE"]
)

print(check["VARIABLE"].value_counts())

In [ ]:
labevents_path = data_path("LABEVENTS.csv.gz")

lab_output_path = data_path("filtered_labevents_17vars.csv")

if os.path.exists(lab_output_path):
    os.remove(lab_output_path)

# HADM_ID -> ICUSTAY_ID
hadm_to_icu = (
    cohort[
        ["HADM_ID", "ICUSTAY_ID"]
    ]
    .dropna()
    .drop_duplicates("HADM_ID")
    .set_index("HADM_ID")["ICUSTAY_ID"]
    .astype(int)
    .to_dict()
)

cohort_hadm = set(hadm_to_icu.keys())

first_write = True
total_kept = 0

reader = pd.read_csv(
    labevents_path,
    usecols=[
        "SUBJECT_ID",
        "HADM_ID",
        "ITEMID",
        "CHARTTIME",
        "VALUENUM",
        "VALUE",
        "VALUEUOM"
    ],
    chunksize=200000,
    low_memory=False
)

for i, chunk in enumerate(reader):

    chunk = chunk.dropna(
        subset=[
            "HADM_ID",
            "ITEMID",
            "CHARTTIME"
        ]
    )

    chunk["HADM_ID"] = pd.to_numeric(
        chunk["HADM_ID"],
        errors="coerce"
    )

    chunk["ITEMID"] = pd.to_numeric(
        chunk["ITEMID"],
        errors="coerce"
    )

    chunk = chunk.dropna(
        subset=["HADM_ID", "ITEMID"]
    )

    chunk["HADM_ID"] = chunk["HADM_ID"].astype(int)
    chunk["ITEMID"] = chunk["ITEMID"].astype(int)

    # cohort admissions
    chunk = chunk[
        chunk["HADM_ID"].isin(cohort_hadm)
    ]

    # benchmark lab ITEMIDs
    chunk = chunk[
        chunk["ITEMID"].isin(lab_itemids)
    ]

    if chunk.empty:
        continue

    chunk["ICUSTAY_ID"] = chunk["HADM_ID"].map(
        hadm_to_icu
    )

    chunk["VARIABLE"] = chunk["ITEMID"].map(
        lab_itemid_to_variable
    )

    chunk.to_csv(
        lab_output_path,
        mode="a",
        header=first_write,
        index=False
    )

    first_write = False
    total_kept += len(chunk)

    if (i + 1) % 20 == 0:
        print(
            f"Processed {(i + 1) * 200000:,} lab rows | "
            f"kept {total_kept:,}"
        )

print("Finished.")
print("Total LABEVENTS kept:", total_kept)